# 🔄 Progressive Windows

## Federated Proactive Forest - Implementación Real de Progressive Windows

Este notebook implementa **EXACTAMENTE** la propuesta teórica de PW con 6 fases:

| Fase | Descripción | Implementación |
|------|-------------|---------------|
| **1** | 🌱 Entrenamiento por ventana | Cada cliente entrena W árboles por ronda |
| **2** | 📤 Comunicación | Clientes envían ventana al servidor |
| **3** | 🔄 Round Robin | Agregación secuencial con score dinámico |
| **4** | 🛑 Progressive Global | Criterio de parada (δ ≤ 0.002 × 2 rondas) |
| **5** | 🔄 Actualización | Clientes incorporan globales (sin duplicados) |
| **6** | 🎯 Inferencia híbrida | ŷ = argmax(λ·p_local + (1-λ)·p_global) |

### 🔁 Flujo Incremental Verdadero

```
RONDA 1:
  ├─ Cliente entrena ventana [0:5] → Envía → Round Robin → Verifica convergencia
  └─ Si NO converge → RONDA 2

RONDA 2:
  ├─ Cliente entrena ventana [5:10] → Envía → Round Robin → Verifica convergencia
  └─ Si NO converge → RONDA 3

... (hasta convergencia o max_rounds)
```

### 📋 Contenido
1. Configuración inicial
2. Carga de datos
3. Hiperparámetros
4. **Ejecución PW Incremental** (logs detallados)
5. Resultados y visualizaciones

---

## 1️⃣ Configuración Inicial

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Librerías cargadas")

---

## 2️⃣ Carga del Dataset

In [ ]:
from pathlib import Path

# Find project root by looking for 'data' directory
current = Path.cwd()
project_root = current
for _ in range(5):  # Look up to 5 levels up
    if (project_root / 'data').exists():
        break
    project_root = project_root.parent

data_path = project_root / 'data' / 'car.csv'
df = pd.read_csv(data_path)
print(f"📊 Dataset: {df.shape[0]} muestras, {df.shape[1]} características")
print(f"\nDistribución Target:\n{df['class'].value_counts()}")
print(f"\nColumnas:\n{df.columns.tolist()}")

In [ ]:
target_column = 'class'
feature_columns = [col for col in df.columns if col != target_column]

# Codificar características categóricas
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder()
X_encoded = encoder.fit_transform(df[feature_columns])

# Codificar target
le = LabelEncoder()
y_encoded = le.fit_transform(df[target_column])
class_names = list(le.classes_)

print(f"\n✅ Features: {X_encoded.shape} | Clases: {class_names}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"\n📈 Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

---

## 3️⃣ Hiperparámetros PW

In [ ]:
import sys
sys.path.append('../../../../')

from src.orchestrators import ProgressiveWindowsOrchestrator
from src.domain.dataset.base_adapter import DatasetSplit

print("✅ PW Incremental importado")

In [ ]:
# =============================================================================
# HIPERPARÁMETROS PW INCREMENTAL
# =============================================================================

config = {
    'n_clients': 5,                  # k: Número de clientes
    'aggregation': {
        'window_size': 5,             # W: Árboles por ventana
        'max_rounds': 20,             # R_MAX: Máximo de rondas
        'alpha': 0.5,                 # α: Balance F1 vs Diversidad
        'convergence_threshold': 0.002  # δ: Umbral de convergencia
    },
    'prediction': {
        'local_weight': 0.5,          # λ: Peso local
        'global_weight': 0.5          # (1-λ): Peso global
    },
    'alpha_pf': 0.1,                  # Parámetro Proactive Forest
    'verbose': True                   # Logs detallados
}

print("=" * 80)
print("⚙️ CONFIGURACIÓN PW INCREMENTAL")
print("=" * 80)
print(f"\n📋 Parámetros:")
print(f"   • Clientes (k): {config['n_clients']}")
print(f"   • Ventana (W): {config['aggregation']['window_size']} árboles")
print(f"   • Máx rondas (R_MAX): {config['aggregation']['max_rounds']}")
print(f"   • Alpha (α): {config['aggregation']['alpha']}")
print(f"   • Lambda (λ): {config['prediction']['local_weight']}")
print(f"   • Convergencia (δ): {config['aggregation']['convergence_threshold']}")
print(f"\n📊 Fórmula Score: Score(T) = α·F1(T) + (1-α)·Diversidad(T|G)")
print(f"📊 Fórmula Inferencia: ŷ = argmax(λ·p_local + (1-λ)·p_global)")
print("=" * 80)

In [ ]:
# Crear DatasetSplit y orquestador
dataset_split = DatasetSplit(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_names=feature_columns,
    class_names=class_names,
    dataset_name='Car Evaluation'
)

# Crear orquestador PW incremental verdadero
orchestrator = ProgressiveWindowsOrchestrator(
    config=config,
    dataset_split=dataset_split,
    verbose=config.get('verbose', True)
)

print(f"\n✅ PW Incremental configurado")
print(f"   • Clientes: {config['n_clients']}")
print(f"   • Ventana (W): {config['aggregation']['window_size']}")
print(f"   • Máx rondas: {config['aggregation']['max_rounds']}")

---

## 4️⃣ EJECUCIÓN PW INCREMENTAL VERDADERO

**IMPORTANTE**: Verás logs detallados de cada ronda:

```
🔁 RONDA X/Y
  🌱 FASE 1: Entrenamiento de ventanas
  🔄 FASE 2-3: Round Robin con ranking
     🎲 Permutación aleatoria
     👤 TURNO 1: client_X
        📊 Ranking: F1, Diversidad, Score
        ✅ SELECCIONADO: T0
  🛑 FASE 4: Criterio de parada
```

**Ejecutar esta celda**

In [ ]:
# Ejecutar PW incremental verdadero
print("\n" + "=" * 100)
print("🚀 INICIANDO PW INCREMENTAL VERDADERO")
print("=" * 100)

results = orchestrator.run_federated_round()

print("\n✅ PW COMPLETADO")

---

## 5️⃣ Resultados

In [ ]:
print("=" * 100)
print("📊 RESULTADOS GLOBALES")
print("=" * 100)

print(f"\n🌲 BOSQUE GLOBAL:")
print(f"   • Árboles: {results.n_trees_global}")
print(f"   • Accuracy: {results.global_accuracy:.6f} ({results.global_accuracy*100:.2f}%)")
print(f"   • Macro-F1: {results.global_macro_f1:.6f}")

print(f"\n🔁 RONDAS:")
print(f"   • Completadas: {results.num_rounds}")
if results.convergence_round and results.convergence_round < results.num_rounds:
    print(f"   • Convergencia temprana: Ronda {results.convergence_round}")
    print(f"   • Rondas ahorradas: {config['aggregation']['max_rounds'] - results.convergence_round}")
else:
    print(f"   • Máximo rondas alcanzado")

print(f"\n📈 ÁRBOLES POR CLIENTE:")
for cid in sorted(results.selected_ids.keys()):
    n = len(results.selected_ids[cid])
    print(f"   • {cid}: {n} árboles")

In [ ]:
# Métricas por cliente
print("\n" + "=" * 100)
print("📈 MÉTRICAS POR CLIENTE (Inferencia Híbrida)")
print("=" * 100)

print(f"\n{'Cliente':<12} | {'Accuracy':<12} | {'Macro-F1':<12} | {'Tamaño':<8}")
print("-" * 100)

for cid in sorted(results.client_ids):
    acc = results.client_accuracies.get(cid, 0)
    f1 = results.client_f1_scores.get(cid, 0)
    size = results.client_hybrid_forest_sizes.get(cid, 0)
    print(f"{cid:<12} | {acc:<12.6f} | {f1:<12.6f} | {size:<8}")

print("-" * 100)
avg_acc = np.mean([results.client_accuracies[cid] for cid in results.client_ids])
avg_f1 = np.mean([results.client_f1_scores[cid] for cid in results.client_ids])
print(f"{'PROMEDIO':<12} | {avg_acc:<12.6f} | {avg_f1:<12.6f} | {'':<8}")

In [ ]:
# Visualización: Árboles seleccionados por cliente
fig, ax = plt.subplots(figsize=(10, 6))

client_ids = sorted(results.selected_ids.keys())
trees_selected = [len(results.selected_ids[cid]) for cid in client_ids]

colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(client_ids)))
bars = ax.bar(client_ids, trees_selected, color=colors, edgecolor='black')

ax.set_xlabel('Cliente')
ax.set_ylabel('Árboles Seleccionados')
ax.set_title('PW Incremental: Árboles Seleccionados por Cliente', fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')

for bar, val in zip(bars, trees_selected):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Matriz de confusión
fig, ax = plt.subplots(figsize=(8, 6))

cm = confusion_matrix(results.y_test, results.global_predictions)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)

ax.set_xlabel('Predicción', fontweight='bold')
ax.set_ylabel('Real', fontweight='bold')
ax.set_title('Matriz de Confusión - Modelo Global PW', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Reporte de clasificación
print("=" * 100)
print("📋 REPORTE DE CLASIFICACIÓN")
print("=" * 100)
print(classification_report(results.y_test, results.global_predictions, target_names=class_names))

In [ ]:
# Resumen final
print("\n" + "=" * 100)
print("📊 RESUMEN FINAL - PW INCREMENTAL")
print("=" * 100)

print(f"\n🔧 HIPERPARÁMETROS:")
print(f"   • Clientes: {config['n_clients']}")
print(f"   • Ventana (W): {config['aggregation']['window_size']}")
print(f"   • Máx rondas: {config['aggregation']['max_rounds']}")
print(f"   • Alpha: {config['aggregation']['alpha']}")
print(f"   • Lambda: {config['prediction']['local_weight']}")

print(f"\n🌲 BOSQUE GLOBAL:")
print(f"   • Árboles: {results.n_trees_global}")
print(f"   • Accuracy: {results.global_accuracy:.4f} ({results.global_accuracy*100:.2f}%)")
print(f"   • Macro-F1: {results.global_macro_f1:.4f}")

print(f"\n🔁 RONDAS:")
print(f"   • Completadas: {results.num_rounds}")
if results.convergence_round and results.convergence_round < results.num_rounds:
    print(f"   • Convergencia: Ronda {results.convergence_round} (temprana)")
else:
    print(f"   • Sin convergencia temprana")

print("\n" + "=" * 100)
print("✅ PW INCREMENTAL COMPLETADO EXITOSAMENTE")
print("=" * 100)